# Etapa 2 — Preparação das imagens (EuroSAT RGB)

Este notebook usa o módulo `notebooks/preprocessamento_py.py` para:

1. Criar os DataLoaders de treino, validação e teste (reutilizando a divisão fixa da Etapa 1, `docs/data_split.json`).
2. Conferir um lote de cada partição (formato, rótulos, média/desvio por canal).
3. Gerar as figuras do relatório em `docs/figuras/`.
4. Provar que validação e teste são determinísticos.

Detalhes e justificativas: `docs/etapa2_preprocessamento.md`.

In [ ]:
# Preparação do ambiente: no Colab clona o repositório; localmente garante que
# estamos na RAIZ do projeto (os caminhos ./data e docs/ são relativos a ela).
import os
import sys

EM_COLAB = "google.colab" in sys.modules

if EM_COLAB:
    if not os.path.exists("/content/Projeto-Visao-computacional"):
        !git clone -b main https://github.com/GabrielGM3/Projeto-Visao-computacional.git /content/Projeto-Visao-computacional
    %cd /content/Projeto-Visao-computacional
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Pasta atual:", os.getcwd())

## 1. Carregar os DataLoaders

Baixe a base explicitamente com `baixar_dataset()`. Na primeira execução isso pode levar alguns minutos; nas seguintes a base local é reutilizada. Importar os módulos não faz downloads.

In [ ]:
import random
import torch

sys.path.append(os.path.abspath("notebooks"))

from notebooks.preprocessamento_py import (
    IMG_SIZE,
    obter_dataloaders,
    get_transforms_treino,
    get_transforms_avaliacao,
    conferir_lote,
    plotar_grade_lote,
    plotar_aumentos,
)

# No notebook do Windows usamos 0 workers; no Colab/Linux usamos 2.
NUM_WORKERS = 0 if os.name == "nt" else 2

from notebooks.dataset_py import baixar_dataset

caminho_dados = baixar_dataset()

treino_dl, val_dl, teste_dl, classes = obter_dataloaders(
    data_dir=caminho_dados,
    img_size=IMG_SIZE,
    batch_size=64,
    num_workers=NUM_WORKERS,
)

print(f"Classes ({len(classes)}): {classes}")
print(f"Imagens -> treino: {len(treino_dl.dataset)} | validação: {len(val_dl.dataset)} | teste: {len(teste_dl.dataset)}")
print(f"Lotes   -> treino: {len(treino_dl)} | validação: {len(val_dl)} | teste: {len(teste_dl)}")

In [ ]:
# Transformações aplicadas em cada partição
print("TREINO:")
print(get_transforms_treino(IMG_SIZE))
print("\nVALIDAÇÃO / TESTE:")
print(get_transforms_avaliacao(IMG_SIZE))

## 2. Conferência dos lotes

O que esperar:
- imagens `[64, 3, 224, 224]` em `float32` e rótulos `int64` entre 0 e 9;
- média e desvio por canal **não** ficam exatamente em 0 e 1: usamos as estatísticas do ImageNet, e as imagens da EuroSAT têm outra distribuição de cores (ex.: no treino o canal R fica com média ≈ -0,55). Isso é esperado — o importante é a rede receber os dados na mesma escala do pré-treino.

No **treino** o lote mistura várias classes (shuffle). Em **validação/teste** não há embaralhamento e o `data_split.json` está ordenado por classe, então o primeiro lote contém só `AnnualCrop` — isso é esperado e não afeta as métricas, que são calculadas sobre a partição inteira.

In [ ]:
print("===== TREINO =====")
_ = conferir_lote(treino_dl, classes)

In [ ]:
print("===== VALIDAÇÃO =====")
_ = conferir_lote(val_dl, classes)

In [ ]:
print("===== TESTE =====")
_ = conferir_lote(teste_dl, classes)

## 3. Figuras para o relatório

### 3.1 Lote de treino (com as variações aleatórias)

In [ ]:
# Sementes fixas só para a figura ser reproduzível (mesmas variações a cada execução)
random.seed(0)
torch.manual_seed(0)

plotar_grade_lote(treino_dl, classes, salvar_em="docs/figuras/lote_treino.png")

### 3.2 Mesma imagem: original x versões aumentadas

Escolhemos uma imagem de `River`, em que giros e espelhamentos são fáceis de perceber.

In [ ]:
treino_ds = treino_dl.dataset

# Primeira imagem da classe River no treino
idx_rio = next(i for i, (_, y) in enumerate(treino_ds.samples) if classes[y] == "River")

# Sementes fixas só para a figura sair igual a cada execução
random.seed(0)
torch.manual_seed(0)

plotar_aumentos(treino_ds, idx=idx_rio, n=5, salvar_em="docs/figuras/aumentos.png")

### 3.3 Lote de validação (sem variações)

O `val_dl` oficial não é embaralhado e o `data_split.json` está ordenado por classe, então o seu primeiro lote teria só `AnnualCrop`. Para a figura, sorteamos uma **amostra fixa de 64 imagens** da validação (semente 42), o que mostra várias classes. As imagens recebem as mesmas transformações de avaliação (sem variações aleatórias).

Isso serve **apenas para visualização**: as métricas de validação e teste continuam sendo calculadas sobre a partição inteira, com o `val_dl` oficial.


In [ ]:
from torch.utils.data import Subset, DataLoader

# Amostra aleatória fixa apenas para visualização; o val_dl oficial continua sem embaralhamento.
val_ds = val_dl.dataset
indices_vis = random.Random(42).sample(range(len(val_ds)), 64)
val_vis_dl = DataLoader(Subset(val_ds, indices_vis), batch_size=64, shuffle=False)

plotar_grade_lote(val_vis_dl, classes, salvar_em="docs/figuras/lote_validacao.png")

## 4. Validação e teste são determinísticos?

Carregamos a mesma imagem duas vezes. Em validação/teste os tensores devem ser **idênticos**; no treino, por causa das variações aleatórias, normalmente são diferentes.

In [ ]:
val_ds = val_dl.dataset
teste_ds = teste_dl.dataset

for nome, ds in [("validação", val_ds), ("teste", teste_ds)]:
    for idx in [0, len(ds) // 2, len(ds) - 1]:
        a, _ = ds[idx]
        b, _ = ds[idx]
        print(f"{nome:<10} idx={idx:<5} iguais? {torch.equal(a, b)}")
        assert torch.equal(a, b), "Validação/teste deveriam ser determinísticos!"

# Duas leituras do primeiro lote do DataLoader de validação
lote1, rot1 = next(iter(val_dl))
lote2, rot2 = next(iter(val_dl))
print(f"\nPrimeiro lote da validação igual nas duas leituras? "
      f"{torch.equal(lote1, lote2) and torch.equal(rot1, rot2)}")

# Para comparação: no treino a mesma imagem muda a cada leitura
iguais_treino = sum(torch.equal(treino_ds[0][0], treino_ds[0][0]) for _ in range(10))
print(f"Treino: leituras repetidas idênticas em {iguais_treino} de 10 tentativas (esperado: poucas ou nenhuma)")

## Resumo

- DataLoaders prontos para as Etapas 3, 4 e 5 via `obter_dataloaders()`.
- Treino: Resize 224 + espelhamentos + giros de 90° + ColorJitter leve + normalização ImageNet.
- Validação/teste: apenas Resize 224 + normalização ImageNet (determinístico, comprovado acima).
- Figuras salvas em `docs/figuras/`: `lote_treino.png`, `aumentos.png`, `lote_validacao.png`.